## 토큰화 (Tokenization) — "문장을 숫자로 번역하기"

> **💡 이 실습의 전체 흐름 (교안 슬라이드 7)**
>
> ```
> "안녕하세요" → [토큰화] → ['안녕', '##하', '##세요'] → [정수 인코딩] → [11655, 2205, 3606]
>     → [임베딩] → [[0.1, 0.4, ...], [0.9, 0.2, ...]]  ← 딥러닝 모델의 진짜 입력
> ```
>
> ⚠️ 위 정수 값은 **형태를 보여주기 위한 예시**이다. 실제 ID는 어떤 토크나이저를 쓰느냐에 따라 완전히 달라진다.
> 잠시 뒤 직접 실행해서 진짜 값을 확인할 것이다.
>
> 딥러닝 모델은 "안녕하세요"라는 글자를 직접 이해할 수 없다.
> 오직 **숫자**만 입력받을 수 있으므로, 텍스트를 숫자로 번역하는 작업이 반드시 선행되어야 한다.
> 이 번역 과정이 **토큰화 → 정수 인코딩 → 임베딩**이다.


In [1]:
# 실습에 필요한 패키지 설치 (최초 1회 실행)
# transformers만 설치하면 안 된다. AutoModel이 내부적으로 PyTorch를 사용하므로 torch도 필요하다.
# (Colab에는 torch가 이미 설치되어 있지만, 로컬 환경에서는 없을 수 있다)
!pip install -q transformers torch



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 실습 1 - 토크나이저 불러오기

In [2]:
from transformers import AutoTokenizer

# ========== 토크나이저 불러오기 ==========
# AutoTokenizer: 모델 이름만 알려주면 해당 모델에 맞는 단어 사전(토크나이저)을
#                자동으로 찾아주는 클래스 (교안 슬라이드 13)
# 'klue/bert-base': 한국어 데이터로 학습된 BERT 모델
#   → 이 모델이 가진 단어 사전과 토큰화 규칙을 그대로 사용
#   → HuggingFace에서 자동 다운로드 (처음만 느리고, 이후 캐시 사용)
tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')

# 토크나이저 정보 출력
print(tokenizer)

# 단어 사전 크기 확인
# vocab_size : 사전 학습 당시에 만들어진 기본 단어 사전의 크기
# len(tokenizer) : 나중에 추가된 토큰까지 포함한 '실제' 전체 크기
#   → 보통 두 값이 같지만, 특수 토큰을 추가하면 달라진다.
#   → 임베딩 층 크기를 정할 때는 반드시 len(tokenizer)를 기준으로 해야 한다.
print(f'\n단어 사전 크기(vocab_size): {tokenizer.vocab_size}')
print(f'실제 전체 크기(len)      : {len(tokenizer)}')


c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--klue--bert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administr

BertTokenizer(name_or_path='klue/bert-base', vocab_size=32000, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

단어 사전 크기(vocab_size): 32000
실제 전체 크기(len)      : 32000


### 실습 2 - 문장 인코딩 및 결과 분석

In [3]:
text = '안녕하세요. 이 실습은 허깅페이스 토크나이저 사용법을 익히는 좋은 예제입니다.'

# ========== 문장 인코딩 (교안 슬라이드 14) ==========
# tokenizer(text): 문장을 모델이 이해할 수 있는 숫자로 변환
# 내부적으로 다음 과정이 자동으로 진행된다:
#   1. 토큰화: 문장을 의미 단위로 분리 ('안녕', '##하', '##세요', ...)
#   2. 정수 인코딩: 각 토큰을 단어 사전에서 찾아 고유 번호(ID)로 변환
#   3. 특수 토큰 추가: 문장 앞에 [CLS], 뒤에 [SEP] 자동 삽입
encoded_input = tokenizer(text)

print(encoded_input)

# 결과는 딕셔너리 형태:
# - input_ids: 각 토큰의 정수 ID 리스트 ← 가장 중요!
# - token_type_ids: 문장 구분 (문장 1개면 전부 0)
# - attention_mask: 실제 단어=1, 패딩(빈칸)=0 (문장 1개면 전부 1)
# → 지금은 input_ids에만 집중하면 된다. 나머지는 이후 챕터에서 다룬다.


{'input_ids': [2, 5891, 2205, 5971, 18, 1504, 10256, 2073, 1905, 2186, 15092, 9157, 7461, 2190, 20650, 2069, 9754, 2259, 1560, 2073, 1439, 2021, 12190, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


### 실습 3 - 토큰 직접 확인하기

In [4]:
# ========== 정수 ID → 토큰 복원 (교안 슬라이드 15) ==========
# input_ids가 정말 원래 문장을 잘 표현하는지, 다시 단어로 되돌려 확인한다.
# convert_ids_to_tokens: 정수 ID → 토큰 문자열
tokens = tokenizer.convert_ids_to_tokens(encoded_input['input_ids'])

print('토큰 목록:', tokens)
print(f'토큰 수: {len(tokens)}')

# 관찰 포인트:
# - [CLS]: 문장의 시작을 알리는 특수 토큰
# - [SEP]: 문장의 끝을 알리는 특수 토큰
# - '##하', '##세요': 서브워드(Subword) — 단어가 더 작은 단위로 나뉜 것
#   → '안녕하세요'가 사전에 통째로 없으면 '안녕' + '##하' + '##세요'로 분해
#   → 이 덕분에 처음 보는 단어(OOV)도 유연하게 처리 가능 (교안 슬라이드 16~17)
#   → '##'은 "앞 토큰에 이어 붙인다"는 의미의 접두사


토큰 목록: ['[CLS]', '안녕', '##하', '##세요', '.', '이', '실습', '##은', '허', '##깅', '##페이스', '토크', '##나이', '##저', '사용법', '##을', '익히', '##는', '좋', '##은', '예', '##제', '##입니다', '.', '[SEP]']
토큰 수: 25


> **🔍 출력에서 꼭 확인할 3가지**
>
> **① `[CLS]` 와 `[SEP]` 이 우리가 넣지 않았는데 생겼다**
> - `[CLS]`: 문장의 시작. 문장 전체를 대표하는 자리로 쓰인다.
> - `[SEP]`: 문장의 끝.
> - 그래서 토큰 수가 원문보다 **2개 많다.**
>
> **② `##` 이 붙은 토큰이 있다**
> `##`는 "앞 토큰에 이어 붙인다"는 표시다. `'안녕' + '##하' + '##세요'`를 이으면 다시 "안녕하세요"가 된다.
>
> **③ 사전에 통째로 있는 단어는 쪼개지지 않는다**
> 자주 쓰이는 단어는 하나의 토큰으로, 드문 단어는 여러 조각으로 나뉜다.
> 👉 **자주 쓰이는 말일수록 토큰 수가 적다.** LLM API가 토큰 단위로 과금하는 이유와 직결된다.

> **🧪 직접 실험해 보기**
>
> 아래를 새 셀에 붙여 실행하며 "어떤 단어가 잘게 쪼개지는가"를 관찰해 보자.
>
> ```python
> for s in ['안녕하세요', '자연어처리', '싸피', 'SSAFY', '킹받네', '나는 학교에 간다']:
>     print(f'{s:12s} -> {tokenizer.tokenize(s)}')
> ```
>
> - 신조어나 영어 고유명사가 어떻게 처리되는지 확인해 보자.
> - 쪼갤 수조차 없으면 `[UNK]`(Unknown)로 처리된다.


In [5]:
for s in ['안녕하세요', '자연어처리', '싸피', 'SSAFY', '킹받네', '나는 학교에 간다']:
    print(f'{s:12s} -> {tokenizer.tokenize(s)}')

안녕하세요        -> ['안녕', '##하', '##세요']
자연어처리        -> ['자연', '##어', '##처리']
싸피           -> ['싸', '##피']
SSAFY        -> ['SS', '##A', '##F', '##Y']
킹받네          -> ['킹', '##받', '##네']
나는 학교에 간다    -> ['나', '##는', '학교', '##에', '간다']


## 임베딩(Embedding) — "숫자를 의미의 좌표로 바꾸기"

> **💡 왜 정수 ID만으로는 부족한가? (교안 슬라이드 20)**
>
> 토큰화로 얻은 정수 ID는 단순한 번호에 불과하다.
> ID 8192("안녕")와 8193("반가워")는 숫자상 가깝지만, **의미적 관계를 전혀 담고 있지 않다.**
> 사물함 번호 101번과 102번이 가깝다고 해서 그 안의 물건이 비슷하지 않은 것과 같다.
>
> 임베딩은 이 단순한 번호를 **의미가 담긴 좌표값(벡터)** 으로 변환하는 과정이다.
> 변환 후에는 의미가 비슷한 단어들이 벡터 공간에서 가까운 위치에 놓이게 된다.

> **📌 자주 인용되는 예시와 그 한계**
>
> 벡터("왕") - 벡터("남자") + 벡터("여자") ≈ 벡터("여왕")
>
> 이 유명한 예시는 **Word2Vec 계열의 정적(static) 임베딩**에서 나온 이야기다.
> 단어 하나당 벡터가 딱 하나로 고정되어 있어서 이런 사칙연산이 성립한다.
>
> 반면 아래 실습에서 BERT로 뽑을 벡터는 **문맥(contextual) 임베딩**이다.
> - 정적 임베딩: "밤"은 언제나 같은 벡터 (야간? 견과류? 구분 불가)
> - 문맥 임베딩: "어두운 **밤**"과 "군**밤** 장수"의 "밤"이 **서로 다른 벡터**
>
> 즉 BERT의 벡터는 문장마다 값이 달라지므로, 위와 같은 단어 사칙연산이 그대로 성립하지는 않는다.
> 두 가지가 다른 개념이라는 점만 지금 기억하고 넘어가자.


### 실습 4 - 모델 불러오기

In [6]:
from transformers import AutoModel

# ========== 모델 불러오기 (교안 슬라이드 23) ==========
# AutoModel: 모델 이름을 알려주면 해당 모델 본체를 자동으로 찾아주는 클래스
#
# 핵심 규칙: 토크나이저와 모델의 이름을 반드시 똑같이 맞춰야 한다! (교안 슬라이드 21)
# → tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')  ← 같은 이름
# → model     = AutoModel.from_pretrained('klue/bert-base')      ← 같은 이름
#
# 왜? klue/bert-base 토크나이저가 '안녕'을 8192번으로 변환했다면,
#     klue/bert-base 모델은 8192번이 어떤 의미인지 학습한 상태이다.
#     다른 모델을 쓰면 8192번을 전혀 다른 단어로 이해할 수 있다.
#     (사람으로 치면 한국어 사전을 보며 영어 문장을 읽는 셈이다)
model = AutoModel.from_pretrained('klue/bert-base')

print(model)

# 참고: from_pretrained()로 불러온 모델은 자동으로 '평가 모드(eval)'로 설정된다.
#       Dropout이 꺼진 상태이므로, 같은 입력을 여러 번 넣어도 항상 같은 결과가 나온다.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9857.18it/s]
[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(32000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

### 실습 5 - 임베딩 벡터 추출하기

In [7]:
import torch

# ========== 임베딩 벡터 추출 (교안 슬라이드 24) ==========

# 1. 인코딩 시 return_tensors='pt' 옵션 추가
# 왜? 앞에서는 결과가 Python 리스트였지만, 모델에 넣으려면 PyTorch 텐서여야 한다.
# 'pt' = PyTorch 텐서 형태로 반환하라는 의미
encoded_input = tokenizer(text, return_tensors='pt')

# 2. 인코딩된 딕셔너리를 모델에 전달
# **encoded_input: 딕셔너리를 풀어서 함수 인자로 전달
# → model(input_ids=..., token_type_ids=..., attention_mask=...)와 동일
#
# torch.no_grad(): 지금은 학습이 아니라 '추론'이므로 미분 정보를 만들 필요가 없다.
#   → 메모리를 아끼고 속도도 빨라진다. 추론 코드에는 습관처럼 붙이자.
with torch.no_grad():
    output = model(**encoded_input)

# 3. 최종 임베딩 벡터 추출
# last_hidden_state: 모델의 마지막 층에서 나온 출력
# 이것이 각 토큰의 "문맥까지 반영된 의미 좌표값"이다.
embedding_vector = output.last_hidden_state

print(f'토큰 수: {encoded_input["input_ids"].shape[1]}')
print(f'임베딩 벡터 shape: {embedding_vector.shape}')
# 출력 예: torch.Size([1, 24, 768])
#   1   = 배치 크기 (문장 1개를 처리)
#   24  = 토큰 수 ([CLS] + 본문 토큰들 + [SEP]) ← 문장에 따라 달라진다
#   768 = 벡터 차원 (하나의 토큰을 768개의 숫자로 표현)
#
# 결론: 문장의 모든 토큰이 각각 768차원의 "의미 벡터"로 변환되었다!
# 이 벡터가 RNN, LSTM, Transformer 등 딥러닝 모델의 진짜 입력이 된다.


토큰 수: 25
임베딩 벡터 shape: torch.Size([1, 25, 768])


> **⚠️ 아주 중요한 구분 — "임베딩"이라는 말이 두 가지 뜻으로 쓰인다**
>
> 앞으로 2_2, 2_3 노트북에서 또 "임베딩"이 나오는데, **방금 뽑은 것과 다른 것**이다.
> 여기서 확실히 구분하지 않으면 반드시 헷갈린다.
>
> | | ① 임베딩 **층** (입력 임베딩) | ② `last_hidden_state` (문맥 임베딩) |
> |---|---|---|
> | 정체 | 단어 사전 크기의 **표(lookup table)** | 모델 12개 층을 **통과한 결과** |
> | 코드 | `model.embeddings.word_embeddings` | `model(**inputs).last_hidden_state` |
> | 문맥 반영 | ❌ 단어당 벡터 1개로 고정 | ✅ 문장마다 값이 달라짐 |
> | 위치 | 모델의 **입구** | 모델의 **출구** |
> | 다루는 노트북 | **2_2, 2_3** | **지금 (2_0)** |
>
> ```
> input_ids ──> [임베딩 층 ①] ──> [Transformer 층 12개] ──> last_hidden_state ②
>                  (입구)                                        (출구)
> ```
>
> 👉 2_2에서 `nn.Embedding`으로 직접 만들 것은 **①번(입구)** 이다.
> 지금 뽑은 ②번은 그 ①번이 12개 층을 거쳐 문맥까지 흡수한 최종 결과물이다.

> **🧪 마지막으로 한 가지만 더 — 문맥 임베딩임을 눈으로 확인하기**
>
> 같은 글자 "밤"이 문장에 따라 다른 벡터가 되는지 직접 확인해 보자.
>
> ```python
> import torch.nn.functional as F
>
> def vec_of(sentence, target):
>     enc = tokenizer(sentence, return_tensors='pt')
>     toks = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
>     idx = toks.index(target)
>     with torch.no_grad():
>         out = model(**enc).last_hidden_state[0, idx]
>     return out
>
> a = vec_of('어두운 밤 하늘에 별이 떴다', '밤')
> b = vec_of('겨울에는 군밤이 최고다', '##밤')
> c = vec_of('깜깜한 밤에 산책을 했다', '밤')
> print('밤(야간) vs 밤(견과류):', F.cosine_similarity(a, b, dim=0).item())
> print('밤(야간) vs 밤(야간)  :', F.cosine_similarity(a, c, dim=0).item())
> ```
>
> 같은 글자인데도 **뜻이 다르면 유사도가 낮게** 나오는 것을 확인할 수 있다.
> 이것이 정적 임베딩이 하지 못하는 일이고, BERT가 강력한 이유다.


In [ ]:
import torch.nn.functional as F


def vec_of(sentence, target):
    enc = tokenizer(sentence, return_tensors='pt')
    toks = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    idx = toks.index(target)
    with torch.no_grad():
        out = model(**enc).last_hidden_state[0, idx]
    return out


a = vec_of('어두운 밤 하늘에 별이 떴다', '밤')
b = vec_of('겨울에는 군밤이 최고다', '##밤')
c = vec_of('깜깜한 밤에 산책을 했다', '밤') 
print('밤(야간) vs 밤(견과류):', F.cosine_similarity(a, b, dim=0).item())  # 0.47607627511024475 # 둘의 벡터값이 같을 확률이 47%
print('밤(야간) vs 밤(야간)  :', F.cosine_similarity(a, c, dim=0).item())  # 0.8550559282302856 # 둘의 벡터값이 같을 확률이 85%

밤(야간) vs 밤(견과류): 0.4760761857032776
밤(야간) vs 밤(야간)  : 0.8550559878349304
